[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_supervised_MA_A.ipynb)

# Part A: the supervised ceiling

The supervised baseline alone: the same CNN-3 trained from scratch with labels, on 1%, 10% and 100% of the Tiny-ImageNet labels, then the frozen evaluation of the 100% encoder (k-NN, linear probe and analysis) on Tiny-ImageNet and CIFAR-100. Its numbers are the reference the DAE is compared with.

These are 9 of the runs of `colab_train_MA_A.ipynb`, with the same commands and the same folder (`midterm_final`), so that notebook skips them once they are finished here. Do not run the two notebooks at the same time: both would train the same runs. About an hour on an L4.

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU. Use an L4 runtime.

In [ ]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

In [ ]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

## Drive
`OUT` is where every run writes.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_final"
os.makedirs(OUT, exist_ok=True)
print("writes to:", OUT)

## TensorBoard
Filter the runs with part of a name (`pretrain`, `linear_probe`, `finetune`, `seed1`), then *Toggle all runs*. The `epoch` section has train, val and test on the same axis; the TEXT tab has each run's `run_info`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

## The runs
The supervised encoder uses the recipe tuned on val for fine-tuning; the linear probe uses the defaults of `config.py`. A finished run (it has `results.json`) is skipped.

In [ ]:
SEED = 42  # the training seed of every run
FRACTIONS = (0.01, 0.1, 1.0)
DATASETS = ("tinyimagenet", "cifar100")
# The recipe tuned on val, as in colab_train_MA_A.ipynb
RECIPE = "--lr 0.003 --batch_size 256 --wd 1e-05 --dropout 0.5 --augment crop_jitter --label_smoothing 0 --optimizer sgd"


def protocol():
    """The supervised encoders, then k-NN, linear probe and analysis of the 100% one."""
    runs = [f"--mode pretrain_supervise_encoder --label_fraction {f} {RECIPE}" for f in FRACTIONS]
    for dataset in DATASETS:
        runs += [f"--mode knn --encoder supervised --dataset {dataset}",
                 f"--mode linear_probe --encoder supervised --dataset {dataset}",
                 f"--mode analysis --encoder supervised --dataset {dataset}"]
    return [f"{flags} --seed {SEED}" for flags in runs]


def run_all(runs):
    for i, flags in enumerate(runs, 1):
        print(f"\n=== {i}/{len(runs)}: {flags}")
        !python train.py {flags} --output_dir "{OUT}"

## Run
9 runs: 3 supervised encoders and 6 evaluations.

In [ ]:
run_all(protocol())

## Results
The supervised runs of `midterm_final`, from their `run_info.txt`.

In [ ]:
import glob

import pandas as pd

rows = []
for path in glob.glob(f"{OUT}/date*supervised*/run_info.txt"):
    with open(path, encoding="utf-8") as f:
        rows.append(dict(line.rstrip("\n").split(": ", 1) for line in f if ": " in line))
runs = pd.DataFrame(rows).sort_values("started")
pd.set_option("display.max_rows", None, "display.max_colwidth", 120)
print(len(runs), "runs")
runs[["started", "folder", "epochs", "results"]]